In [70]:
# 1. načíst výstup modelu
# 2. načíst 48 prvních samplů z gt
# 3. udělat list v délce samplu z modelu, index je pozice tokenu v stringu
# 4. porovnat stringy, tam kde jsou stejné dát na daný index +1

In [ ]:
import json
import os
import json
from glob import glob
from transformers import PreTrainedTokenizerFast

def load_step_jsons(base_path):
    """
    Load all JSON files from step folders in the given base path.
    
    Args:
        base_path (str): Path to the directory containing step folders
        
    Returns:
        list: List of tuples containing (step_number, json_data)
    """
    results = []
    
    # Get all step directories using glob
    step_dirs = glob(os.path.join(base_path, "step_*"))
    for step_dir in step_dirs:
        try:
            # Extract step number from directory name
            step_num = int(os.path.basename(step_dir).split('_')[1])
            
            # Path to json file in this step directory
            json_path = os.path.join(step_dir, "results_48_sos_filtered_val1_b4_t30_n500000_dfs_filtered.json")
            
            # Read and parse JSON
            with open(json_path, 'r') as f:
                data = json.load(f)
                
            results.append((step_num, data))
            
        except ValueError as e:
            print(f"Error parsing step number from {step_dir}: {e}")
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON in step {step_num}: {e}")
        except Exception as e:
            print(f"Error processing {step_dir}: {e}")
    
    # Sort results by step number
    results.sort(key=lambda x: x[0])
    
    return results

base_path = "../data/eval_results/Pythia-12-8-128-dfs"
jsons = load_step_jsons(base_path)

print(f"Loaded {len(jsons)} JSON files")

eval_data = []
with open("/mnt/raid/data/Hyner_Petr/rl/litgpt/rl_basic_transformer/data/sos_filtered/val1_b4_t30_n500000_dfs_filtered.json", "r") as f:
    for line in f:
        if line.strip():
            try:
                eval_data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line: {e}")
                continue

def get_tokenizer(path):
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=path
    )
    tokenizer.eos_token = "[EOS]"
    tokenizer.unk_token = "[UNK]"
    tokenizer.pad_token = "[PAD]"
    tokenizer.mask_token = "[MASK]"
    tokenizer.bos_token = "[BOS]"
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


In [72]:
tokenizer = get_tokenizer("/mnt/raid/data/Hyner_Petr/rl/litgpt/rl_basic_transformer/tokenizer/tokenizer.json")

def compare_lists(list1, list2):
    # Get the length of the longer list
    max_length = max(len(list1), len(list2))
    
    # Initialize result list with zeros
    result = [0] * max_length
    
    # Compare elements at corresponding indices
    for i in range(min(len(list1), len(list2))):
        if list1[i] == list2[i]:
            result[i] = 1
            
    return result

def get_eval_ids_lst(eval_data, num_samples):
    ret = []
    for i, sample in enumerate(eval_data):
        if i < num_samples:
            sp = sample["search_path"][10:]
            encoded_2 = tokenizer.encode(sp, return_tensors='pt')
            ret.append(encoded_2.tolist()[0])
    return ret

eval_input_ids = get_eval_ids_lst(eval_data, 48)
from collections import defaultdict
full_dic = defaultdict(list)
for json in jsons:
    ix = 0
    step_num = json[0]
    print(step_num)
    for traj in json[1]["trajectories"]:
        traj = traj.replace("[BOS] ", "")[10:]
        encoded = tokenizer.encode(traj, return_tensors='pt')
        out_ids_lst = encoded.tolist()[0]
        result = compare_lists(out_ids_lst, eval_input_ids[ix])
        count_matches = result.count(1)
        match_indices = [i for i, value in enumerate(result) if value == 1]
        tokenized_out = tokenizer.tokenize(traj)
        match_toks = []
        for index in match_indices:
            match_toks.append(tokenized_out[index])

        full_dic[step_num].append((ix, count_matches, match_indices, match_toks))
        ix+=1

print(full_dic[8334])




In [ ]:
aggr = []
for k, v in full_dic.items():
    all_matches = 0
    for i, res in enumerate(v):
        ix, num_matches, indices, tokens = res
        all_matches += num_matches
    aggr.append(all_matches/i)
print(aggr)

In [74]:
toks = []
for k, v in full_dic.items():
    toks_temp = []
    for i, res in enumerate(v):
        _, _, _, tokens = res
        toks_temp.append(tokens)
    toks.append(toks_temp)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data=aggr
# Create figure and axis
plt.figure(figsize=(15, 6))

# Create bar plot
plt.bar(range(len(data)), data, color='skyblue', edgecolor='black')

# Customize the plot
plt.title('Bar Plot')
plt.xlabel('Index')
plt.ylabel('Value')

# Add grid for better readability
plt.grid(True, alpha=0.3, axis='y')

# Display the plot
plt.tight_layout()
plt.show()

In [76]:
from collections import Counter
ret = defaultdict(list)
for ix, json in enumerate(toks):
    dic = Counter()  # Use Counter directly instead of regular dict
    for lst in json:
        dic.update(Counter(lst))  # Update will sum the counts
    
    ret[ix].append(dict(dic))

In [ ]:
len(ret)

In [ ]:
ret[4]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_histogram(index):
    # Get the dictionary for the specified index
    data = ret[index][0]  # ret[index] is a list containing one dict
    
    # Create the histogram
    plt.figure(figsize=(15, 6))
    plt.bar(range(len(data)), list(data.values()))
    
    # Set the x-axis labels (tokens)
    plt.xticks(range(len(data)), list(data.keys()), rotation=45, ha='right')
    
    plt.title(f'Token Counts for Dictionary {index}')
    plt.xlabel('Tokens')
    plt.ylabel('Count')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example usage:
# plot_histogram(0)  # Plot histogram for index 0
# plot_histogram(57)  # Plot histogram for index 57

# If you want to create an interactive plot where you can switch indices:
from ipywidgets import interact

@interact(index=(0, 57))
def plot_interactive_histogram(index):
    plot_histogram(index)

In [ ]:
# S 15 [ 23 19 15 13 ] , E 23 - 13 = 10 R [ 23 19 15 13 ] , G #00 15 [ 23 19 15 13 ] , M #00 , S 15 [ 19 15 10 ] , E 19 - 10 = 9 R [ 19 15 10 ] , G #000 15 [ 19 15 10 ] , M #000 , S 15 [ 15 9 ]

full_dic[8334]

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def create_highlighted_text(step_num, traj_idx):
    text = eval_data[traj_idx]["search_path"][10:]
    text_lst = text.split(" ")
    
    # Access data exactly like your script
    _, _, match_indices, _ = full_dic[step_num][traj_idx]
    match_indices.sort()
    
    # Create HTML with highlights
    html_parts = []
    for idx, word in enumerate(text_lst):
        if idx in match_indices:
            html_parts.append(f'<span style="background-color: red; color: white;">{word}</span>')
        else:
            html_parts.append(word)
    
    highlighted_text = ' '.join(html_parts)
    return HTML(f'<div style="font-family: monospace; white-space: pre-wrap; font-size: 14px;">{highlighted_text}</div>')

def interactive_text_viewer():
    initial_step = 8334  # Set initial step to 8334
    if full_dic[0]:
        del full_dic[0]
    step_dropdown = widgets.Dropdown(
        options=sorted(full_dic.keys()),
        value=initial_step,  # Set initial value
        description='Step:',
        style={'description_width': 'initial'}
    )
    
    def update_traj_options(*args):
        traj_dropdown.options = range(len(full_dic[step_dropdown.value]))
    
    traj_dropdown = widgets.Dropdown(
        options=range(len(full_dic[initial_step])),  # Use initial_step here too
        description='Trajectory:',
        style={'description_width': 'initial'}
    )
    
    step_dropdown.observe(update_traj_options, 'value')
    
    output = widgets.Output()
    
    def update_display(*args):
        with output:
            output.clear_output()
            display(create_highlighted_text(
                step_dropdown.value,
                traj_dropdown.value
            ))
    
    step_dropdown.observe(update_display, 'value')
    traj_dropdown.observe(update_display, 'value')
    
    display(widgets.HBox([step_dropdown, traj_dropdown]))
    display(output)
    update_display()

# Run the viewer
interactive_text_viewer()